# YOLOv8n Fine-Tuning on CrowdHuman for Dense Person Detection

This Colab notebook converts CrowdHuman ODGT annotations into YOLO format, trains YOLOv8n with requested hyperparameters, validates performance, exports ONNX, and runs a quick inference demo.

In [ ]:
# --- SETUP ---
# Install required libraries exactly as requested for model training and dataset handling.
%pip install ultralytics datasets huggingface_hub Pillow

# Import core libraries for filesystem operations, parsing annotations, image utilities, and training.
import os
import json
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

from ultralytics import YOLO

# Define reusable paths used throughout the notebook.
CROWDHUMAN_ROOT = Path('/content/crowdhuman')
YOLO_ROOT = Path('/content/crowdhuman_yolo')
IMAGES_ROOT = YOLO_ROOT / 'images'
LABELS_ROOT = YOLO_ROOT / 'labels'
MODELS_ROOT = Path('/content/models')
TEST_OUTPUT_ROOT = Path('/content/yolo_test_outputs')

# Ensure output directories exist before writing files.
for p in [YOLO_ROOT, IMAGES_ROOT, LABELS_ROOT, MODELS_ROOT, TEST_OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('Runtime paths initialized.')
print(f'CrowdHuman root: {CROWDHUMAN_ROOT}')
print(f'YOLO dataset root: {YOLO_ROOT}')
print(f'Model export root: {MODELS_ROOT}')

In [ ]:
# --- DATA CONVERSION HELPERS ---
# Resolve image path robustly from CrowdHuman sample ID by checking common extensions and folders.
def resolve_crowdhuman_image_path(crowdhuman_root: Path, image_id: str) -> Path | None:
    candidate_rel_dirs = [Path('images'), Path('train'), Path('val'), Path('.')]
    candidate_exts = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    for rel_dir in candidate_rel_dirs:
        for ext in candidate_exts:
            p = crowdhuman_root / rel_dir / f'{image_id}{ext}'
            if p.exists() and p.is_file():
                return p
    return None

# Create a symlink when possible for speed and storage savings, otherwise copy as fallback.
def symlink_or_copy(src: Path, dst: Path) -> None:
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)

# Convert one absolute [x, y, w, h] full-body box to normalized YOLO [cx, cy, w, h].
def xywh_to_yolo(x: float, y: float, w: float, h: float, img_w: int, img_h: int):
    cx = (x + (w / 2.0)) / float(img_w)
    cy = (y + (h / 2.0)) / float(img_h)
    nw = w / float(img_w)
    nh = h / float(img_h)

    # Clamp values to [0, 1] bounds to keep labels valid.
    cx = max(0.0, min(1.0, cx))
    cy = max(0.0, min(1.0, cy))
    nw = max(0.0, min(1.0, nw))
    nh = max(0.0, min(1.0, nh))
    return cx, cy, nw, nh

In [ ]:
# --- DATA CONVERSION ---
# Convert CrowdHuman ODGT annotations to YOLO format for either train or val split.
def convert_crowdhuman_to_yolo(
    annotation_path: Path,
    split_name: str,
    max_images: int,
    crowdhuman_root: Path = CROWDHUMAN_ROOT,
    yolo_root: Path = YOLO_ROOT,
):
    # Create split-specific image and label directories.
    split_images_dir = yolo_root / 'images' / split_name
    split_labels_dir = yolo_root / 'labels' / split_name
    split_images_dir.mkdir(parents=True, exist_ok=True)
    split_labels_dir.mkdir(parents=True, exist_ok=True)

    if not annotation_path.exists():
        print(f'[WARN] Missing annotation file: {annotation_path}')
        return {
            'split': split_name,
            'images_converted': 0,
            'annotations_converted': 0,
            'avg_persons_per_image': 0.0,
        }

    images_converted = 0
    annotations_converted = 0

    with annotation_path.open('r', encoding='utf-8') as f:
        for line_idx, line in enumerate(f):
            if images_converted >= max_images:
                break

            line = line.strip()
            if not line:
                continue

            # Each ODGT line is a standalone JSON object.
            obj = json.loads(line)
            image_id = str(obj.get('ID', f'img_{line_idx:06d}'))
            gtboxes = obj.get('gtboxes', [])

            image_src = resolve_crowdhuman_image_path(crowdhuman_root, image_id)
            if image_src is None:
                continue

            # Read image size to normalize bounding boxes correctly.
            with Image.open(image_src) as img:
                img_w, img_h = img.size

            # Collect valid YOLO label lines for this image.
            yolo_lines = []
            for ann in gtboxes:
                fbox = ann.get('fbox', None)
                if fbox is None or len(fbox) != 4:
                    continue

                x, y, w, h = float(fbox[0]), float(fbox[1]), float(fbox[2]), float(fbox[3])

                # Skip invalid boxes as required (w <= 0 or h <= 0).
                if w <= 0.0 or h <= 0.0:
                    continue

                cx, cy, nw, nh = xywh_to_yolo(x, y, w, h, img_w, img_h)

                # Keep only strictly positive normalized width/height boxes.
                if nw <= 0.0 or nh <= 0.0:
                    continue

                # YOLO format line: class_id cx cy w h (single class: person -> class 0).
                yolo_lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')

            # Skip images with no valid person boxes.
            if len(yolo_lines) == 0:
                continue

            # Link/copy image into YOLO images split folder.
            image_dst = split_images_dir / image_src.name
            symlink_or_copy(image_src, image_dst)

            # Write one label text file per image.
            label_dst = split_labels_dir / f'{image_dst.stem}.txt'
            with label_dst.open('w', encoding='utf-8') as lf:
                lf.write('\n'.join(yolo_lines) + '\n')

            images_converted += 1
            annotations_converted += len(yolo_lines)

    avg_persons = (annotations_converted / images_converted) if images_converted > 0 else 0.0

    print(f'[{split_name}] total images converted: {images_converted}')
    print(f'[{split_name}] total annotations: {annotations_converted}')
    print(f'[{split_name}] average persons per image: {avg_persons:.2f}')

    return {
        'split': split_name,
        'images_converted': images_converted,
        'annotations_converted': annotations_converted,
        'avg_persons_per_image': avg_persons,
    }

# Create Ultralytics dataset YAML with requested keys and values.
def create_yaml_config(yaml_path: Path = YOLO_ROOT / 'dataset.yaml') -> Path:
    yaml_text = (
        'path: /content/crowdhuman_yolo\n'
        'train: images/train\n'
        'val: images/val\n'
        'nc: 1\n'
        "names: ['person']\n"
    )
    yaml_path.write_text(yaml_text, encoding='utf-8')
    print(f'Wrote dataset YAML to: {yaml_path}')
    return yaml_path

# Run train and val conversion with requested caps for Colab runtime control.
train_stats = convert_crowdhuman_to_yolo(
    annotation_path=CROWDHUMAN_ROOT / 'annotation_train.odgt',
    split_name='train',
    max_images=8000,
)
val_stats = convert_crowdhuman_to_yolo(
    annotation_path=CROWDHUMAN_ROOT / 'annotation_val.odgt',
    split_name='val',
    max_images=2000,
)

# Create dataset config consumed by Ultralytics training API.
dataset_yaml = create_yaml_config()

# Print combined conversion summary.
total_images = train_stats['images_converted'] + val_stats['images_converted']
total_anns = train_stats['annotations_converted'] + val_stats['annotations_converted']
avg_persons = (total_anns / total_images) if total_images > 0 else 0.0
print('--- Combined conversion summary ---')
print(f'Total images converted: {total_images}')
print(f'Total annotations: {total_anns}')
print(f'Average persons per image: {avg_persons:.2f}')

In [ ]:
# --- TRAINING ---
# Load YOLOv8n pretrained backbone to start fine-tuning from strong generic visual features.
model = YOLO('yolov8n.pt')

# Train with exact requested parameters, each commented for project-report clarity.
train_results = model.train(
    data='/content/crowdhuman_yolo/dataset.yaml',  # Dataset definition file for train/val splits and classes
    epochs=30,                                     # Number of full passes over training data
    imgsz=640,                                     # Input image resolution balancing detail and speed
    batch=16,                                      # Batch size chosen for Colab GPU memory constraints
    lr0=0.001,                                     # Initial learning rate for stable fine-tuning
    lrf=0.01,                                      # Final LR ratio for cosine-style schedule inside Ultralytics
    momentum=0.937,                                # SGD momentum-like term used by YOLO optimizer
    weight_decay=0.0005,                           # L2 regularization to reduce overfitting
    warmup_epochs=3,                               # Warmup period to stabilize early optimization
    box=7.5,                                       # Box regression loss gain (localization emphasis)
    cls=0.5,                                       # Classification loss gain (single class still supervised)
    dfl=1.5,                                       # Distribution Focal Loss gain for box quality
    device=0,                                      # Use first CUDA GPU in Colab (as requested)
    project='/content/yolo_runs',                  # Root folder for training artifacts
    name='crowdhuman_finetune',                    # Run name for reproducible experiment tracking
    exist_ok=True,                                 # Allow reuse of run folder if it already exists
    plots=True,                                    # Save training curves and debug visualizations
    save=True,                                     # Save checkpoints including best model
    val=True                                       # Run validation during training
)

In [ ]:
# --- POST-TRAIN VALIDATION + EXPORT ---
# Run validation after training and print requested mAP metrics.
val_results = model.val()

# Extract mAP values from Ultralytics metrics object.
map50 = float(val_results.box.map50)
map5095 = float(val_results.box.map)
print(f'mAP@0.5: {map50:.4f}')
print(f'mAP@0.5:0.95: {map5095:.4f}')

# Export ONNX model for deployment portability across runtimes.
onnx_export_path = model.export(format='onnx', dynamic=True, simplify=True)
print(f'ONNX export path: {onnx_export_path}')

# Resolve best checkpoint path produced by training run.
best_pt_candidate = Path('/content/yolo_runs/crowdhuman_finetune/weights/best.pt')
if best_pt_candidate.exists():
    best_pt_path = best_pt_candidate
else:
    # Fallback to trainer metadata if available.
    trainer_best = getattr(getattr(model, 'trainer', None), 'best', None)
    best_pt_path = Path(str(trainer_best)) if trainer_best else None

if best_pt_path is None or not best_pt_path.exists():
    raise FileNotFoundError('Could not locate best.pt after training.')

# Copy best model to required output location.
final_best_path = Path('/content/models/yolo_crowdhuman_best.pt')
final_best_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_pt_path, final_best_path)
print(f'Saved best model to: {final_best_path}')

In [ ]:
# --- QUICK INFERENCE TEST ---
# Load final best model for post-training qualitative verification.
best_model = YOLO('/content/models/yolo_crowdhuman_best.pt')

# Gather validation images from converted YOLO dataset.
val_image_dir = Path('/content/crowdhuman_yolo/images/val')
val_images = sorted([p for p in val_image_dir.glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']])
if len(val_images) == 0:
    raise RuntimeError('No validation images found for inference test.')

# Select up to 3 random images for quick visual sanity check.
num_samples = min(3, len(val_images))
sample_paths = random.sample(val_images, k=num_samples)

output_dir = Path('/content/yolo_test_outputs')
output_dir.mkdir(parents=True, exist_ok=True)

for i, img_path in enumerate(sample_paths, start=1):
    # Run prediction on one validation image.
    preds = best_model.predict(source=str(img_path), verbose=False)
    result = preds[0]

    # Load image with OpenCV for manual box drawing.
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy() if result.boxes is not None else np.zeros((0, 4))

    # Draw predicted person boxes using cv2.rectangle as requested.
    for b in boxes:
        x1, y1, x2, y2 = [int(v) for v in b.tolist()]
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    print(f'Image {i}: {img_path.name} -> persons detected: {len(boxes)}')

    out_path = output_dir / f'annotated_{img_path.stem}.jpg'
    cv2.imwrite(str(out_path), img)

print(f'Annotated outputs saved to: {output_dir}')